In [1]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [2]:
SYSTEM_PROMPT_STRING = """
You are an NYT Connections solver.
You will be given 16 words.
OUTPUT ONLY the final groups of words and their categories.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups and their categories.
Format EXACTLY like this:

GROUP 1: word1, word2, word3, word4
CATEGORY: category_name

GROUP 2: word1, word2, word3, word4
CATEGORY: category_name

GROUP 3: word1, word2, word3, word4
CATEGORY: category_name

GROUP 4: word1, word2, word3, word4
CATEGORY: category_name
"""

In [3]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = ", ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split(",")] for group in groups ]
    
    return parsed_groups

In [4]:
from conn import load_connections_from_hf

hf_split = load_connections_from_hf()
print("puzzles:", len(hf_split))

c:\Users\kunri\miniconda3\envs\cs175\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


puzzles: 652


In [11]:
def solve_puzzle(words16, k=0, split_for_few_shot=None, model="llama3.1-8b", temperature=0.1, max_tokens=600):
    few_shot_prompt = ""
    if k > 0 and split_for_few_shot is not None:
        few_shot_prompt += "Here are some previous examples:"
        count = 0
        for fs_row in split_for_few_shot:
            fs_words = fs_row["words"]
            # Skip because of leakage
            if set(fs_words) == set(words16):
                continue
            fs_groups = [ans["words"] for ans in fs_row["answers"]]
            fs_text = f"Here are 16 words: {', '.join(fs_words)}\n"
            for i, group in enumerate(fs_groups, 1):
                fs_text += f"GROUP {i}: {', '.join(group)}\n"
            few_shot_prompt += fs_text + "\n"
            count += 1
            if count >= k:
                break

    user_prompt = few_shot_prompt + "NOW, solve this puzzle and only this one: " + convert_puzzle_to_prompt(words16)
    response = call_llm(SYSTEM_PROMPT_STRING, user_prompt, model=model, 
                        temperature=temperature, max_tokens=max_tokens)
    return parse_response_to_pred_groups(response)

In [12]:
# Pick the first puzzle
row0 = hf_split[6]
words16 = row0["words"]

# Solve it with zero few-shot examples
pred_groups = solve_puzzle(words16, k=10)

# Display predicted groups
print("Predicted groups:")
for i, g in enumerate(pred_groups, 1):
    print(f"GROUP {i}: {g}")

# Display gold groups for comparison
print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])

Predicted groups:
GROUP 1: ['EASY', 'OPEN', 'RECEPTIVE', 'FLEXIBLE']
GROUP 2: ['EVIL', 'VILE', 'VEIL', 'WICKED']
GROUP 3: ['AMAZING', 'GENIUS', 'SOLID', 'LIT']
GROUP 4: ['LIVE', 'BEGINNER', 'SCENTED', 'WAXY']

Gold groups:
AMENABLE -> ['EASY', 'FLEXIBLE', 'OPEN', 'RECEPTIVE']
ANAGRAMS -> ['EVIL', 'LIVE', 'VEIL', 'VILE']
SPELLING BEE RANKS -> ['AMAZING', 'BEGINNER', 'GENIUS', 'SOLID']
ADJECTIVES FOR A CANDLE -> ['LIT', 'SCENTED', 'WAXY', 'WICKED']


In [ ]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    evaluate,
    gold_groups_from_row,
)

N_EVAL = 10

few_shot_values = [0, 2, 5, 10, 20]
for k in few_shot_values:
    #Split to avoid leakage
    indices = list(range(k, len(hf_split)))
    split_subset = hf_split.select(indices)
    few_shot_subset = hf_split.select(list(range(k)))

    results = evaluate(
        split_subset,  #use only puzzles after the first k
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=few_shot_subset),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=True,
        verbose_every=2
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Processed 2/10 samples
Processed 4/10 samples
Processed 6/10 samples
Processed 8/10 samples
Processed 10/10 samples
k=0 | Zero-one accuracy: 0.4444  (n=9, requested=10)
k=0 | Mean 1-1 swaps to correct: 2.00  (n=9)
Processed 2/10 samples
Processed 4/10 samples
Processed 6/10 samples
Processed 8/10 samples
Processed 10/10 samples
k=2 | Zero-one accuracy: 0.8000  (n=10, requested=10)
k=2 | Mean 1-1 swaps to correct: 0.40  (n=10)
Processed 2/10 samples
Processed 4/10 samples
Processed 6/10 samples
Processed 8/10 samples
Processed 10/10 samples
k=5 | Zero-one accuracy: 0.8000  (n=10, requested=10)
k=5 | Mean 1-1 swaps to correct: 0.20  (n=10)
Processed 2/10 samples
Processed 4/10 samples
Processed 6/10 samples
Processed 8/10 samples
Processed 10/10 samples
k=10 | Zero-one accuracy: 1.0000  (n=10, requested=10)
k=10 | Mean 1-1 swaps to correct: 0.00  (n=10)
Processed 2/10 samples
Processed 4/10 samples
Processed 6/10 samples
Processed 8/10 samples
Processed 10/10 samples
k=20 | Zero-one accu